# ChaoSSM — Phase 13c: the six-system regime control at n = 5, with paired reporting

This is Phase 13b re-run at **five seeds instead of three**, writing to its own
`results/phase13c_ordered_control_n5/` folder so the earlier n = 3 results are left intact.

**Why re-run.** At n = 3 a single collapsed seed carries a third of the mass, and the cubic and sine
families each produced one (accuracy ~0.05-0.15 while the other two seeds ran at 0.6+). Reporting a mean
over a bimodal set of three is not defensible. Five seeds does two things: it drops a single collapse to a
fifth of the mass, and it makes the collapse *rate* itself something we can state rather than apologize for.

**Why five and not eight.** The load-bearing claim here is ordered-versus-edge, and that contrast is
**paired by seed** — both arms share the initialization, the input stream and the eval set, and the
collapses appear in both arms of the same seed, so they largely cancel in the difference. Five paired
differences is adequate for a directional claim that supports an n = 20 single-system result; it is not
being asked to carry the argument alone. Going to eight would buy little and cost ~60% more GPU time.

**What is added beyond more seeds.** The analysis now reports, per family: the median alongside the mean,
the paired per-seed ordered-minus-edge difference with a bootstrap CI, the number of seeds each arm wins,
and an explicit count of collapsed runs (accuracy below the linear control) so the failure mode is
reported rather than averaged away.


**Purpose.** Reviewer 1's architectural-confound concern (major point 1) applies not only to the
headline logistic result (addressed in the Phase 14 notebook) but to the generality claim across
chaotic systems. This notebook adds the **regime-pure `ordered` control (lambda ~= -0.30)** to the
frozen-reservoir generality experiment, for every system, so that the three-point dynamical-regime axis
— **ordered / edge** — is available across logistic, tent, cubic, sine, Rössler, and Lorenz, not only
for the logistic map. If the edge-of-chaos reservoir beats BOTH the linear reservoir AND the ordered
(same-architecture, non-critical) reservoir across structurally unrelated systems, the benefit is
attributable to the critical dynamical regime rather than to the specific map family or to the
reservoir architecture.

This notebook reuses, **verbatim**, the validated Phase 13 v2 machinery: the chaotic maps and ODE-flow
cells (Rössler/Lorenz integrated as recurrent cells), the ReservoirChaosCell / LinearCellRes classes,
the gamma-calibration with validation, and the Experiment-B frozen-reservoir harness. The ONLY change
is the addition of the `ordered` lambda target and a second regime in the Experiment-B loop. Experiment
A (the suppression/gated end-to-end comparison) is not needed here and is omitted to keep the run cheap.


## Setup (verbatim from Phase 13 v2)

In [ ]:
import os, json, math, time, warnings
from dataclasses import dataclass, asdict, field
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
import pandas as pd
warnings.filterwarnings("ignore")
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch",torch.__version__,"| device:",DEVICE)
if DEVICE.type=="cuda": print("gpu:",torch.cuda.get_device_name(0))
def setup_results_dir(project="ChaoSSM",phase="phase13c_ordered_control_n5"):
    """Resolve the output directory, preferring Google Drive so results survive the runtime.

    Colab's drive.mount() can fail (e.g. ValueError when /content/drive already holds files, or
    when the mount is refused). Earlier versions silently fell back to the VM's local disk, where
    results are lost when the runtime recycles. We now retry the mount once with force_remount and,
    if Drive is still unavailable, fall back to local disk with a loud warning."""
    base=None
    try:
        from google.colab import drive
        try:
            drive.mount("/content/drive",force_remount=False)
        except Exception as e1:
            print(f"[warn] drive.mount failed ({type(e1).__name__}: {e1}); retrying with force_remount=True")
            drive.mount("/content/drive",force_remount=True)
        base=f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base=os.path.abspath(f"/content/{project}_results/{phase}") if os.path.isdir("/content") \
             else os.path.abspath(f"./{project}_results/{phase}")
        print("="*78)
        print(f"[WARNING] Google Drive is NOT mounted ({type(e).__name__}).")
        print(f"[WARNING] Results will be written to {base}")
        print("[WARNING] That is the runtime's local disk -- the files are LOST when the runtime")
        print("[WARNING] recycles. Download them, or copy them to Drive, as soon as the run ends.")
        print("="*78)
    os.makedirs(base,exist_ok=True); print("results ->",base); return base
RESULTS=setup_results_dir()
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=300,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)
def savejson(o,n):
    p=os.path.join(RESULTS,n)
    with open(p,"w") as fh: json.dump(o,fh,indent=2,default=str)
    print("saved",p)

## Configuration — Phase 13 v2 verbatim, plus the `ordered` regime target

In [ ]:
SMOKE_TEST=False

@dataclass
class Config:
    state_dim:int=64; d_model:int=64; n_layers:int=2; mlp_mult:int=2
    steps_A:int=3000     # experiment A (suppression, gated, end-to-end)
    steps_B:int=2500     # experiment B (frozen reservoir, readout-only -- cheap)
    batch:int=64; warmup_frac:float=0.05; lr_min_frac:float=0.1
    grad_clip:float=1.0; lr:float=1e-3
    seeds:int=5
    eval_n:int=1200; eval_seed:int=12345
    copy_len:int=12; vocab:int=16
    fc_length:int=64; fc_bins:int=32
    lambda_target:dict=field(default_factory=lambda: dict(ordered=-0.30, edge=-0.02, deep=0.30))
    lambda_tol:float=0.06; calib_steps:int=150; calib_warm:int=50
    benettin_warmup:int=80; benettin_steps:int=300; benettin_traj:int=10
    map_families:tuple=("logistic","tent","cubic","sine","rossler","lorenz")

CFG=Config()
if SMOKE_TEST:
    CFG.steps_A=40; CFG.steps_B=30; CFG.seeds=2; CFG.eval_n=40
    CFG.calib_steps=30; CFG.calib_warm=10; CFG.benettin_warmup=10; CFG.benettin_steps=30; CFG.benettin_traj=4
    print("*** SMOKE TEST ***")
print(json.dumps({k:v for k,v in asdict(CFG).items()},indent=2,default=str))

## Chaotic maps + ODE flows (verbatim from Phase 13 v2)

In [ ]:
def triangle_fold(x): return 1.0-torch.abs(torch.remainder(x,2.0)-1.0)

# ---- Rössler and Lorenz: continuous ODE flows, not elementwise maps -------------
# Unlike logistic/tent/cubic/sine (which act independently on each scalar state
# dimension), these are 3-variable coupled systems. We partition the state into
# groups of 3 dimensions, rescale each triple from [0,1] into the system's natural
# chaotic-attractor range, integrate forward with a fixed number of RK4 substeps at
# the classical chaotic parameter values, then rescale back into [0,1]. Any leftover
# dimensions (state_dim not divisible by 3) receive no flow contribution and are
# driven purely by the W_mix coupling term, exactly as the other families' local_map
# contributes zero to a dimension only through r; this is a minor, disclosed
# simplification for this generality check, not used in the main results.
_LORENZ_BOUNDS  = torch.tensor([[-20.,20.],[-25.,25.],[0.,50.]])   # (x,y,z) native range
_ROSSLER_BOUNDS = torch.tensor([[-12.,12.],[-12.,12.],[0.,25.]])

def _lorenz_rhs(s):
    x,y,z = s[...,0],s[...,1],s[...,2]
    sigma,rho,beta = 10.0,28.0,8.0/3.0
    return torch.stack([sigma*(y-x), x*(rho-z)-y, x*y-beta*z], dim=-1)

def _rossler_rhs(s):
    x,y,z = s[...,0],s[...,1],s[...,2]
    a,b,c = 0.2,0.2,5.7
    return torch.stack([-y-z, x+a*y, b+z*(x-c)], dim=-1)

def _rk4_step(rhs, s, dt):
    k1=rhs(s); k2=rhs(s+0.5*dt*k1); k3=rhs(s+0.5*dt*k2); k4=rhs(s+dt*k3)
    return s+(dt/6.0)*(k1+2*k2+2*k3+k4)

def _flow_contribution(h, rhs, bounds, n_sub=4, dt=0.02):
    N=h.shape[-1]; n_triples=N//3; rem=N-3*n_triples
    dev=h.device; bounds=bounds.to(dev)
    if n_triples==0: return torch.zeros_like(h)
    h_triples=h[...,:3*n_triples].reshape(*h.shape[:-1],n_triples,3)
    lo=bounds[:,0]; hi=bounds[:,1]
    native=lo+h_triples*(hi-lo)
    s=native
    for _ in range(n_sub): s=_rk4_step(rhs,s,dt)
    s=torch.clamp(s,lo-1e-3,hi+1e-3)  # guard against rare overshoot
    normalized=(s-lo)/(hi-lo)
    contrib_triples=normalized-h_triples   # increment, O(1) scale like the other maps
    contrib=contrib_triples.reshape(*h.shape[:-1],3*n_triples)
    if rem>0:
        contrib=torch.cat([contrib,torch.zeros_like(h[...,3*n_triples:])],dim=-1)
    return contrib

def local_map(h, r, family):
    if family=="logistic": return r*h*(1-h)
    if family=="tent":     return r*torch.minimum(h, 1-h)
    if family=="cubic":    return r*h*(1-h*h)
    if family=="sine":     return r*torch.sin(math.pi*h)/math.pi
    if family=="lorenz":   return _flow_contribution(h,_lorenz_rhs,_LORENZ_BOUNDS)
    if family=="rossler":  return _flow_contribution(h,_rossler_rhs,_ROSSLER_BOUNDS)
    raise ValueError(family)

class ChaoticMap(nn.Module):
    def __init__(self,N,family,r_cap=4.0,gi=1.0,learnable_gamma=True):
        super().__init__(); self.family=family; self.r_cap=float(r_cap)
        self.w_r=nn.Parameter(torch.zeros(N))
        W=torch.empty(N,N); nn.init.orthogonal_(W); self.W_mix=nn.Parameter(W*0.1)
        wg=torch.tensor(float(math.log(math.expm1(gi))))
        self.w_gamma=nn.Parameter(wg) if learnable_gamma else wg
        self.learnable_gamma=learnable_gamma
    def get_r(self): return self.r_cap*torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def set_gamma(self,val):
        with torch.no_grad():
            new=torch.tensor(float(math.log(math.expm1(max(val,1e-3)))))
            if self.learnable_gamma: self.w_gamma.data.copy_(new)
            else: self.w_gamma=new
    def forward(self,h):
        r=self.get_r(); u=h@self.W_mix.T+local_map(h,r,self.family)
        return triangle_fold(self.get_gamma()*u)

## Reservoir cells: ReservoirChaosCell / LinearCellRes (verbatim from Phase 13 v2)

In [ ]:
class GatedChaosCell(nn.Module):
    def __init__(self,cfg,family):
        super().__init__(); self.N=cfg.state_dim
        self.phi=ChaoticMap(cfg.state_dim,family,r_cap=4.0,gi=1.0,learnable_gamma=True)
        self.A=nn.Parameter(torch.diag(torch.empty(cfg.state_dim).uniform_(0.9,1.1)))
        self.B=nn.Parameter(torch.randn(cfg.state_dim,cfg.state_dim)/math.sqrt(cfg.state_dim))
        self.W_g=nn.Parameter(torch.randn(cfg.state_dim,cfg.state_dim)/math.sqrt(cfg.state_dim))
        self.b_g=nn.Parameter(torch.zeros(cfg.state_dim))
    def step(self,h,x):
        g=torch.sigmoid(x@self.W_g.T+self.b_g)
        raw=g*self.phi(h@self.A.T)+(1-g)*h+x@self.B.T
        return triangle_fold(raw)
    def init_state(self,b,dev,dt): return torch.full((b,self.N),0.5,device=dev,dtype=dt)

class ReservoirChaosCell(nn.Module):
    # No gate (fair-test style); B may be frozen (canonical RC) via freeze_input().
    def __init__(self,cfg,family,regime):
        super().__init__(); self.N=cfg.state_dim; self.regime=regime
        self.phi=ChaoticMap(cfg.state_dim,family,r_cap=4.0,gi=1.0,learnable_gamma=True)
        self.B=nn.Parameter(torch.randn(cfg.state_dim,cfg.state_dim)/math.sqrt(cfg.state_dim))
    def step(self,h,x): return triangle_fold(self.phi(h)+x@self.B.T)
    def init_state(self,b,dev,dt): return torch.full((b,self.N),0.5,device=dev,dtype=dt)
    def freeze_recurrence(self):
        for p_ in [self.phi.w_r,self.phi.W_mix,self.phi.w_gamma]: p_.requires_grad_(False)
    def freeze_input(self): self.B.requires_grad_(False)

class LinearCellRes(nn.Module):
    def __init__(self,cfg,**kw):
        super().__init__(); self.N=cfg.state_dim
        W=torch.empty(self.N,self.N); nn.init.orthogonal_(W); self.W_lin=nn.Parameter(W*0.9)
        self.B=nn.Parameter(torch.randn(cfg.state_dim,cfg.state_dim)/math.sqrt(cfg.state_dim))
    def step(self,h,x): return torch.tanh(h@self.W_lin.T+x@self.B.T)
    def init_state(self,b,dev,dt): return torch.zeros(b,self.N,device=dev,dtype=dt)
    def freeze_recurrence(self): self.W_lin.requires_grad_(False)
    def freeze_input(self): self.B.requires_grad_(False)

## Tasks, gamma-calibration, validated Benettin/QR estimator (verbatim from Phase 13 v2)

In [ ]:
V=CFG.vocab; SEP,PAD=V,V+1; VOCAB_SIZE=V+2
def make_copy(cfg,batch,device,gen):
    L=cfg.copy_len; src=torch.randint(0,V,(batch,L),device=device,generator=gen)
    inp=torch.cat([src,torch.full((batch,1),SEP,device=device),torch.full((batch,L),PAD,device=device)],1)
    tgt=torch.cat([torch.full((batch,L+1),-100,device=device),src],1)
    return inp,tgt,"tokens",VOCAB_SIZE,VOCAB_SIZE
def make_forecast(cfg,batch,device,gen):
    L=cfg.fc_length; r=3.9
    x0=0.1+0.8*torch.rand(batch,1,device=device,generator=gen); xs=[x0]
    for _ in range(L): xs.append(r*xs[-1]*(1-xs[-1]))
    X=torch.cat(xs,1); inp=X[:,:L]; nxt=X[:,1:L+1]
    tgt=torch.clamp((nxt*cfg.fc_bins).long(),0,cfg.fc_bins-1)
    return inp.float(),tgt,"scalar",1,cfg.fc_bins
TASK_FN=dict(copy=make_copy,forecast=make_forecast)
@torch.no_grad()
def majority_class_chance(cfg,task,n=15000,seed=777):
    g=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); g.manual_seed(seed)
    _,tgt,_,_,_=TASK_FN[task](cfg,n,DEVICE,g); m=tgt!=-100
    nb=cfg.fc_bins if task=="forecast" else cfg.vocab
    counts=torch.bincount(tgt[m].reshape(-1),minlength=nb)
    return float(counts.max())/float(counts.sum())
CHANCE={t: majority_class_chance(CFG,t) for t in ("copy","forecast")}
print("chance:",{k:round(v,4) for k,v in CHANCE.items()})

def build_eval(cfg,task):
    g=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); g.manual_seed(cfg.eval_seed)
    return TASK_FN[task](cfg,cfg.eval_n,DEVICE,g)
@torch.no_grad()
def eval_acc(model,evalset,chunk=256):
    inp,tgt,_,_,_=evalset; model.eval(); c=tot=0.0
    for i in range(0,inp.shape[0],chunk):
        lg=model(inp[i:i+chunk]); tg=tgt[i:i+chunk]; m=tg!=-100; p=lg.argmax(-1)
        c+=((p==tg)&m).float().sum().item(); tot+=m.float().sum().item()
    model.train(); return c/max(tot,1)

from torch.func import jacrev
@torch.no_grad()
def map_lyapunov(cell,cfg,seed=0):
    steps=cfg.calib_steps; warm=cfg.calib_warm; N=cfg.state_dim; dev=cell.W_mix.device
    g=torch.Generator(device=dev); g.manual_seed(seed)
    h=torch.full((N,),0.5,device=dev); Q=torch.linalg.qr(torch.randn(N,N,generator=g,device=dev))[0]
    logsum=torch.zeros(N,device=dev)
    for t in range(steps+warm):
        def stp(z): return triangle_fold(cell.get_gamma()*(z@cell.W_mix.T+local_map(z,cell.get_r(),cell.family)))
        J=jacrev(stp)(h); h=stp(h)+0.02*torch.randn(N,generator=g,device=dev); h=torch.clamp(h,0,1)
        Q,R=torch.linalg.qr(J@Q); d=torch.sign(torch.diagonal(R)); Q=Q*d; R=R*d.unsqueeze(0)
        if t>=warm: logsum=logsum+torch.log(torch.abs(torch.diagonal(R))+1e-12)
    return float(torch.sort(logsum/steps,descending=True)[0][0])

def calibrate_gamma(cell,cfg,regime,seed=0,max_restarts=3):
    # Bisect gamma to hit the target lambda, with validation: if bisection fails to converge
    # within tolerance (can happen for maps whose lambda(gamma) is non-monotone or noisy near the
    # target), retry with a fresh random seed for the driven-perturbation trajectory used inside
    # map_lyapunov, up to max_restarts times, before accepting the best available fit. Returns
    # (gamma, achieved_lambda, calibration_ok) so callers can flag/exclude failed calibrations
    # rather than silently averaging over them.
    target=cfg.lambda_target[regime]
    best_overall=None; best_overall_err=1e9
    for attempt in range(max_restarts):
        lo,hi=0.05,10.0; best=None; best_err=1e9
        for _ in range(20):
            mid=math.sqrt(lo*hi); cell.set_gamma(mid)
            lam=map_lyapunov(cell,cfg,seed=seed*97+attempt*13)
            err=abs(lam-target)
            if err<best_err: best_err=err; best=(mid,lam)
            if lam<target: lo=mid
            else: hi=mid
            if err<cfg.lambda_tol: break
        if best_err<best_overall_err: best_overall_err=best_err; best_overall=best
        if best_err<cfg.lambda_tol: break
    cell.set_gamma(best_overall[0])
    calibration_ok = bool(best_overall_err < 2*cfg.lambda_tol)
    return best_overall[0], best_overall[1], calibration_ok

def logistic_lyap(r=4.0,n=3000,warm=200,seed=0):
    rng=np.random.default_rng(seed); x=rng.uniform(0.1,0.9); s=0.0; c=0
    for t in range(n+warm):
        d=abs(r*(1-2*x)); x=r*x*(1-x)
        if t>=warm: s+=math.log(d+1e-12); c+=1
    return s/max(c,1)
KNOWN=logistic_lyap(); print(f"[validation] logistic r=4 lambda={KNOWN:.4f} vs ln2={math.log(2):.4f} err={abs(KNOWN-math.log(2)):.4f}")

## Conditional-lambda estimator (verbatim from Phase 13 v2)

In [ ]:
@torch.no_grad()
def real_input_sequence(model,cfg,seed,layer,length,task,in_kind,is_reservoir):
    g=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); g.manual_seed(500+seed)
    inp,_,_,_,_=TASK_FN[task](cfg,1,DEVICE,g)
    e = model.emb(inp) if in_kind=="tokens" else model.inp(inp.unsqueeze(-1))
    dt=e.dtype; T=inp.shape[1]
    if is_reservoir:
        cell=model.cell; h=cell.init_state(1,DEVICE,dt); cap=[]
        for t in range(T):
            u=e[:,t]; cap.append(u.squeeze(0).detach()); h=cell.step(h,u)
    else:
        blk=model.blocks[layer]; h=blk.init_state(1,DEVICE,dt); cap=[]
        for t in range(T):
            u=blk.ln1(e[:,t]); cap.append(u.squeeze(0).detach())
            hh=blk.cell.step(h,u); y=e[:,t]+hh; y=y+blk.mlp(blk.ln2(y)); h=hh; e[:,t]=y if False else e[:,t]
        # (kept minimal: gated-model conditional lambda not needed for family-generality claim beyond A)
    seq=torch.stack(cap,0)
    if seq.shape[0]<length: seq=seq.repeat((length//seq.shape[0])+1,1)
    return seq[:length]

@torch.no_grad()
def conditional_lambda_reservoir(model,cfg,seed,task,in_kind):
    cell=model.cell; N=cfg.state_dim; total=cfg.benettin_warmup+cfg.benettin_steps
    lam_maxes=[]
    for tr in range(cfg.benettin_traj):
        g=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); g.manual_seed(9000+seed*97+tr)
        seqx=real_input_sequence(model,cfg,9000+seed*97+tr,0,total,task,in_kind,True)
        h=torch.full((N,),0.5,device=DEVICE)
        Q=torch.linalg.qr(torch.randn(N,N,generator=g,device=DEVICE))[0]; logsum=torch.zeros(N,device=DEVICE)
        for t in range(total):
            x=seqx[t%seqx.shape[0]]
            J=jacrev(lambda z: cell.step(z.unsqueeze(0),x.unsqueeze(0)).squeeze(0))(h)
            h=cell.step(h.unsqueeze(0),x.unsqueeze(0)).squeeze(0)
            Q,R=torch.linalg.qr(J@Q); d=torch.sign(torch.diagonal(R)); Q=Q*d; R=R*d.unsqueeze(0)
            if t>=cfg.benettin_warmup: logsum=logsum+torch.log(torch.abs(torch.diagonal(R))+1e-12)
        lam_maxes.append(float(torch.sort(logsum/cfg.benettin_steps,descending=True)[0][0]))
    return float(np.mean(lam_maxes))

## Experiment B+ — ordered vs edge frozen reservoir across all systems

Identical to the Phase 13 v2 Experiment-B harness, with two changes only:
1. the regime loop is now `("ordered","edge")` instead of `("edge",)`;
2. nothing else — same cell, same calibration, same canonical readout-only protocol, same forecast
   task, same linear-reservoir control, same seed discipline.

The `set_results_dir` writes to its own `phase13c_ordered_control_n5/` folder and does not touch any
existing Phase 13 results.

In [ ]:
class ReservoirLM(nn.Module):
    def __init__(self,cfg,cell,out_dim):
        super().__init__(); self.inp=nn.Linear(1,cfg.d_model); self.cell=cell
        self.ln=nn.LayerNorm(cfg.state_dim); self.readout=nn.Linear(cfg.state_dim,out_dim)
    def forward(self,x):
        e=self.inp(x.unsqueeze(-1)); B,T=e.shape[0],e.shape[1]; dt=e.dtype
        h=self.cell.init_state(B,e.device,dt); outs=[]
        for t in range(T):
            h=self.cell.step(h,e[:,t]); outs.append(self.readout(self.ln(h)))
        return torch.stack(outs,1)

def make_lr_lambda(cfg,total):
    warm=max(1,int(cfg.warmup_frac*total))
    def fn(s):
        if s<warm: return s/warm
        p=(s-warm)/max(1,total-warm); return cfg.lr_min_frac+(1-cfg.lr_min_frac)*0.5*(1+math.cos(math.pi*p))
    return fn

# ---------------------------------------------------------------------------------------
# RESUMABLE RUN.
# The first attempt at this sweep lost ~40 min of completed work when the Colab runtime
# disconnected before the single end-of-run save. Two changes fix that permanently:
#   1. every finished run is appended to the CSV immediately (checkpoint after each run),
#      so a disconnect costs at most the run in flight;
#   2. on startup any existing CSV is loaded and the (family, regime, seed) combinations it
#      already contains are SKIPPED, so re-running this cell continues where it stopped.
# To resume an interrupted sweep, just put the partial phase13c_ordered_benefit.csv in the
# results folder and run this cell again. To force a clean restart, delete that file (or set
# FRESH_START=True below).
# ---------------------------------------------------------------------------------------
FRESH_START=False

CKPT=os.path.join(RESULTS,"phase13c_ordered_benefit.csv")
COLS=["family","regime","arm","seed","diverged","acc","calibrated_lambda","calibration_ok"]

if FRESH_START and os.path.exists(CKPT):
    os.replace(CKPT,CKPT+".bak"); print("[resume] FRESH_START -> previous CSV moved to",CKPT+".bak")

if os.path.exists(CKPT):
    rowsB=pd.read_csv(CKPT).to_dict("records")
    done={(r["family"],r["regime"],int(r["seed"])) for r in rowsB}
    print(f"[resume] found {len(rowsB)} completed runs in {CKPT}")
    have=sorted({r['family'] for r in rowsB})
    print("[resume] families already present:",have)
else:
    rowsB=[]; done=set(); print("[resume] no checkpoint found -- starting from scratch")

def _flush():
    pd.DataFrame(rowsB)[COLS].to_csv(CKPT,index=False)

planned=[]
for fam in CFG.map_families:
    for reg in ("ordered","edge"):
        planned += [(fam,reg,sd) for sd in range(CFG.seeds)]
    planned += [(fam,"linear-control",sd) for sd in range(CFG.seeds)]
todo=[k for k in planned if k not in done]
print(f"[resume] {len(planned)} runs planned, {len(planned)-len(todo)} already done, {len(todo)} to run\n")

t0=time.time()
for family,regime,sd in todo:
    if regime=="linear-control":
        torch.manual_seed(1000+sd); np.random.seed(1000+sd)
        gen=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(1000+sd)
        cell=LinearCellRes(CFG).to(DEVICE); lam_cal=float("nan"); cal_ok=True
    else:
        torch.manual_seed(sd); np.random.seed(sd)
        gen=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(sd)
        cell=ReservoirChaosCell(CFG,family,regime).to(DEVICE)
        gam,lam_cal,cal_ok=calibrate_gamma(cell.phi,CFG,regime,seed=sd)
        if not cal_ok:
            print(f"    [calibration WARNING] {family}/{regime} seed={sd}: achieved lambda={lam_cal:+.3f} "
                  f"missed target={CFG.lambda_target[regime]:+.3f} beyond tolerance -- flagging, not excluding")
    cell.freeze_recurrence(); cell.freeze_input()
    model=ReservoirLM(CFG,cell,CFG.fc_bins).to(DEVICE)
    params=[p for p in model.parameters() if p.requires_grad]
    opt=torch.optim.AdamW(params,lr=CFG.lr,betas=(0.9,0.95))
    sched=torch.optim.lr_scheduler.LambdaLR(opt,make_lr_lambda(CFG,CFG.steps_B))
    div=False
    for step in range(CFG.steps_B):
        inp,tgt,_,_,_=TASK_FN["forecast"](CFG,CFG.batch,DEVICE,gen)
        lg=model(inp); L=F.cross_entropy(lg.reshape(-1,lg.shape[-1]),tgt.reshape(-1))
        opt.zero_grad(set_to_none=True); L.backward()
        gn=torch.nn.utils.clip_grad_norm_(params,CFG.grad_clip).item(); opt.step(); sched.step()
        if not (math.isfinite(L.item()) and math.isfinite(gn)): div=True; break
    acc=eval_acc(model,build_eval(CFG,"forecast")) if not div else float("nan")
    rowsB.append(dict(family=family,regime=regime,arm=("linear" if regime=="linear-control" else "chaos"),
                      seed=sd,diverged=div,acc=acc,calibrated_lambda=lam_cal,calibration_ok=cal_ok))
    _flush()   # checkpoint after every single run
    tag=("linear         " if regime=="linear-control" else f"chaos-{regime:7s}")
    lamtxt="" if regime=="linear-control" else f" lam_cal={lam_cal:+.3f}"
    print(f"[B+] {family:9s} {tag} s={sd} acc={acc:.4f}{lamtxt}"+(" DIV" if div else ""))

RESB=pd.DataFrame(rowsB)[COLS]
_flush()
print(f"\nExperiment B+ wall clock this session: {(time.time()-t0)/60:.1f} min")
print(f"total runs now in {CKPT}: {len(RESB)}")

## Analysis: ordered vs edge vs linear, per system

In [ ]:
import numpy as np
LINEAR=RESB[RESB.regime=="linear-control"].groupby("family").acc.mean()

def boot_ci(v,seed=0,n=10000):
    v=np.asarray(v,float); v=v[np.isfinite(v)]
    if len(v)<2: return (float("nan"),float("nan"))
    rng=np.random.default_rng(seed); b=rng.choice(v,size=(n,len(v)),replace=True).mean(1)
    return float(np.percentile(b,2.5)),float(np.percentile(b,97.5))

rows=[]
for family in CFG.map_families:
    d=RESB[RESB.family==family]
    edge=d[d.regime=="edge"].sort_values("seed").acc.values
    ordd=d[d.regime=="ordered"].sort_values("seed").acc.values
    linv=float(LINEAR.get(family,float("nan")))
    k=min(len(edge),len(ordd))
    paired=edge[:k]-ordd[:k]                      # same seed, same init, same eval set
    plo,phi=boot_ci(paired,seed=1)
    # a run is called collapsed if it falls below the linear control, i.e. the nonlinear reservoir
    # has not merely underperformed but failed; reported, never excluded
    n_coll=int((edge<linv).sum()+(ordd<linv).sum())
    rows.append(dict(system=family,
                     ordered_mean=round(float(np.nanmean(ordd)),3),
                     ordered_median=round(float(np.nanmedian(ordd)),3),
                     edge_mean=round(float(np.nanmean(edge)),3),
                     edge_median=round(float(np.nanmedian(edge)),3),
                     linear_mean=round(linv,3),
                     edge_minus_ordered=round(float(np.nanmean(paired)),3),
                     paired_ci=f"[{plo:+.3f},{phi:+.3f}]",
                     edge_wins=int((edge[:k]>ordd[:k]).sum()), n_seeds=k,
                     collapsed_runs=n_coll))
SUMM=pd.DataFrame(rows); print(SUMM.to_string(index=False)); savetable(SUMM,"phase13c_summary.csv")

print("\nPer-seed accuracies (so the bimodality is visible rather than averaged away):")
for family in CFG.map_families:
    d=RESB[RESB.family==family]
    o=" ".join(f"{a:.3f}" for a in d[d.regime=="ordered"].sort_values("seed").acc.values)
    e=" ".join(f"{a:.3f}" for a in d[d.regime=="edge"].sort_values("seed").acc.values)
    print(f"  {family:9s} ordered: {o}   edge: {e}")

n_edge_better=int((SUMM.edge_minus_ordered>0).sum())
print(f"\nR1#1 generality check: edge is ahead of the same-architecture ordered reservoir on "
      f"{n_edge_better}/{len(SUMM)} systems (paired, per seed).")
print("-> edge ahead on most systems with CIs excluding zero would tie the benefit to the critical")
print("   regime; edge at or below ordered indicates the benefit belongs to the stable nonlinear")
print("   regime instead, consistent with the n=20 single-system control of Phase 14.")
print(f"   collapsed runs (below the linear control): {int(SUMM.collapsed_runs.sum())} of "
      f"{len(CFG.map_families)*2*CFG.seeds} reservoir runs")

## Figure — ordered vs edge vs linear across systems (300 DPI)

In [ ]:
fig,ax=plt.subplots(figsize=(11,4.6))
systems=list(CFG.map_families); x=np.arange(len(systems)); w=0.26
ordv=[SUMM[SUMM.system==s].ordered_mean.values[0] for s in systems]
edgev=[SUMM[SUMM.system==s].edge_mean.values[0] for s in systems]
linv=[SUMM[SUMM.system==s].linear_mean.values[0] for s in systems]
ax.bar(x-w,ordv,w,label="ordered (λ≈-0.30)",color="#4C72B0",edgecolor="white")
ax.bar(x, edgev,w,label="edge (λ≈0⁻)",color="#2E7D32",edgecolor="white")
ax.bar(x+w,linv,w,label="linear reservoir",color="#9E9E9E",edgecolor="white")
ax.set_xticks(x); ax.set_xticklabels([s.capitalize() for s in systems],fontsize=11)
ax.set_ylabel("Forecasting accuracy\n(frozen reservoir, canonical protocol)",fontsize=11)
ax.set_title("Regime-pure control across chaotic systems: edge vs ordered vs linear",fontsize=12)
ax.legend(fontsize=10,ncol=3); ax.grid(alpha=0.25,axis="y")
fig.tight_layout(); savefig(fig,"phase13c_ordered_axis.png"); plt.close(fig)
print("figure saved")


## Rescue the results if Drive was not mounted


In [ ]:
# --- save-guard: if Drive was not mounted, make the results easy to rescue -------------
import shutil, os
if not RESULTS.startswith("/content/drive"):
    _zip = shutil.make_archive("/content/phase13c_results","zip",RESULTS)
    print("\n" + "="*78)
    print("Drive was not mounted, so these results live only on this runtime's local disk.")
    print("A zip has been written to:", _zip)
    print("Download it now (Files pane, or run the two lines below) before the runtime recycles:")
    print("  from google.colab import files")
    print(f"  files.download('{_zip}')")
    print("="*78)
else:
    print("\nresults are on Drive:", RESULTS)